# Polyp segmentation: Kaggle training notebook

**Before you run:** Settings (right side) -> **Accelerator: GPU T4 x2**, **Internet: ON**. Add the input dataset *medico-automatic-polyp-segmentation-challenge* (by debeshjha1) with **+ Add Input**.

**How to use (3 modes, set `MODE` in the settings cell):**
1. `MODE = "timing"` -> trains 2 epochs of fold 0 for each variant and prints the real speed. Run this first (about 10 minutes).
2. `MODE = "train"` -> full training of every fold in `FOLDS` for every variant in `VARIANTS`, then evaluates.
3. `MODE = "eval"` -> only evaluates runs that are already finished.

Long runs: use **Save Version -> Save & Run All (Commit)** so it keeps running without your browser. Everything is resumable: finished folds are skipped. To continue in a new session, put the previous version's output `runs` folder path in `RESUME_FROM`.

In [ ]:
# ===================== EDIT ONLY THIS CELL =====================
GIT_URL  = "https://github.com/KUNALSHAWW/Transformer-Based-Robust-Polyp-Segmentation-for-Cross-Dataset-Colonoscopy-Analysis.git"
BRANCH   = "paper-research"
PROTOCOL = "A"          # "A" = train Kvasir-880, zero-shot tests | "B" = standard PraNet protocol | "Bclean" = standard, ClinicDB split by video
VARIANTS = ["base"]     # base, roy_plain, roy, roy_size, roy_overlay, roy_full, roy_overlay_aug, roy_size_gate_only, roy_size_loss_only
SEEDS    = [0]
FOLDS    = [0, 1, 2, 3, 4]
EPOCHS   = 60
WORKERS  = 2
BS       = 8            # lower to 4 if you see an out-of-memory error
MODE     = "timing"     # "timing" | "train" | "eval"
RESUME_FROM = ""        # e.g. "/kaggle/input/<your-previous-notebook-output>/runs"
# ===============================================================

In [ ]:
import os, sys, glob, json, time, shutil, subprocess, zipfile
def sh(cmd):
    print("$", cmd); subprocess.run(cmd, shell=True, check=True)
sh("pip install -q segmentation-models-pytorch albumentations timm")
if os.path.exists("/kaggle/working/code"): shutil.rmtree("/kaggle/working/code")
sh(f"git clone -q --depth 1 -b {BRANCH} {GIT_URL} /kaggle/working/code")
CODE = "/kaggle/working/code/polyp_paper"; sys.path.insert(0, CODE)
sh("git -C /kaggle/working/code log -1 --format='code version: %h %s'")
import torch
print("GPUs visible:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# Build the data folder (Kvasir-SEG from the attached input; ClinicDB, ColonDB, ETIS, CVC-300 and PraNet's split are downloaded)
DATA = "/kaggle/working/data"
cands = [c for c in glob.glob("/kaggle/input/**/Kvasir-SEG", recursive=True) if os.path.isdir(os.path.join(c, "images"))]
assert cands, "Kvasir-SEG not found: add the input dataset 'medico-automatic-polyp-segmentation-challenge' (+ Add Input)"
if not os.path.exists(DATA + "/splits"):
    sh(f'{sys.executable} {CODE}/prepare_data.py --out {DATA} --kvasir "{cands[0]}" --download')
print("data ready:", os.listdir(DATA))
sh(f"cd {CODE} && {sys.executable} splits.py --data_root {DATA} --protocol {PROTOCOL}")

In [ ]:
# Quick safety check of the code (about 1 minute). It must end with: SELF-CHECK PASSED
sh(f"cd {CODE} && {sys.executable} selfcheck.py --data_root {DATA}")

In [ ]:
# ---------- launcher: runs jobs on both GPUs at the same time, skips finished folds ----------
RUNS = "/kaggle/working/runs"; os.makedirs(RUNS, exist_ok=True)
if RESUME_FROM and os.path.exists(RESUME_FROM):
    shutil.copytree(RESUME_FROM, RUNS, dirs_exist_ok=True); print("resumed from", RESUME_FROM)
OUT = "/kaggle/working/timing" if MODE == "timing" else RUNS
os.makedirs(OUT, exist_ok=True)
done = lambda v, s, f: os.path.exists(f"{RUNS}/{PROTOCOL}/{v}/seed{s}/fold{f}/manifest.json")
if MODE == "timing":
    jobs, ep = [(v, SEEDS[0], 0) for v in VARIANTS], 2
elif MODE == "train":
    jobs, ep = [(v, s, f) for v in VARIANTS for s in SEEDS for f in FOLDS if not done(v, s, f)], EPOCHS
else:
    jobs, ep = [], EPOCHS
print(len(jobs), "job(s):", jobs)
ngpu = max(1, torch.cuda.device_count()); running = {}; t_start = time.time(); tick = 0
def tail(path, n=1):
    try: return open(path).read().strip().split("\n")[-n:]
    except Exception: return []
while jobs or running:
    for g in range(ngpu):
        if g not in running and jobs:
            v, s, f = jobs.pop(0)
            log = f"{OUT}/log_{PROTOCOL}_{v}_s{s}_f{f}.txt"
            cmd = [sys.executable, "train.py", "--data_root", DATA, "--protocol", PROTOCOL, "--variant", v, "--fold", str(f), "--seed", str(s),
                   "--out", OUT, "--epochs", str(ep), "--workers", str(WORKERS), "--bs", str(BS)]
            p = subprocess.Popen(cmd, cwd=CODE, env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)), stdout=open(log, "w"), stderr=subprocess.STDOUT)
            running[g] = (p, v, s, f, time.time(), log); print(f"[{(time.time()-t_start)/60:6.1f} min] start  {v} seed{s} fold{f} on GPU {g}")
    time.sleep(20); tick += 1
    for g, (p, v, s, f, t0, log) in list(running.items()):
        if p.poll() is not None:
            print(f"[{(time.time()-t_start)/60:6.1f} min] {'OK  ' if p.returncode == 0 else 'FAILED'} {v} seed{s} fold{f} ({(time.time()-t0)/60:.1f} min)")
            if p.returncode != 0: print("   last log lines:", tail(log, 5))
            del running[g]
    if tick % 15 == 0:
        for g, (p, v, s, f, t0, log) in running.items(): print("   progress:", tail(log)[0][:140])
print("all jobs finished in %.1f min" % ((time.time() - t_start) / 60))

In [ ]:
# ---------- timing report (only for MODE = "timing") ----------
if MODE == "timing":
    for v in VARIANTS:
        h = json.load(open(f"{OUT}/{PROTOCOL}/{v}/seed{SEEDS[0]}/fold0/history.json"))
        steady = h[-1]["sec"]                                   # last epoch = steady speed (epoch 1 includes warm-up)
        fold_min = steady * 38 / 60                             # earlier runs stopped after about 38 epochs on average
        print(f"{v:20s} {steady:5.0f} s/epoch | about {fold_min:4.0f} min per fold | about {fold_min * 5 / 60:3.1f} GPU-hours per 5-fold run")
        print("                     validation Dice per epoch:", [round(e["val_dice"], 4) for e in h], "(must change from epoch to epoch and differ between versions)")
    print("Note these numbers: they decide how many runs fit into your weekly GPU hours.")

In [ ]:
# ---------- evaluation on the TEST sets (threshold comes from out-of-fold predictions only) ----------
EVAL = "/kaggle/working/eval"
if MODE in ("train", "eval"):
    for v in VARIANTS:
        for s in SEEDS:
            ok = all(done(v, s, f) for f in range(5))
            if not ok:
                print("skip", v, "seed", s, "(not all 5 folds finished)"); continue
            sh(f"cd {CODE} && {sys.executable} evaluate.py --data_root {DATA} --protocol {PROTOCOL} --variant {v} --seed {s} --runs {RUNS} --out {EVAL} --overlay_eval")

In [ ]:
# ---------- package the small result files (no checkpoints) ----------
bundle = "/kaggle/working/results_bundle.zip"
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for root in ("/kaggle/working/runs", "/kaggle/working/eval", "/kaggle/working/timing"):
        for dp, _, fs in os.walk(root):
            for f in fs:
                if f.endswith(".pt") or f == "oof_probs.npz": continue      # large files stay on Kaggle (evaluate.py reads them there)
                p = os.path.join(dp, f); z.write(p, os.path.relpath(p, "/kaggle/working"))
print(bundle, round(os.path.getsize(bundle) / 1e6, 1), "MB  <- download this file from the Output tab and keep it with your logbook")